# RSNA 2024 Lumbar Spine – LumbarNet eğitimi

**Kaggle ayarları (sağ panel):**
1. *Add Input* → **RSNA 2024 Lumbar Spine Degenerative Classification** yarışma verisini ekleyin (önce yarışma sayfasında kuralları kabul etmelisiniz).
2. *Accelerator* → **GPU T4 x2** veya **P100**.
3. *Internet* → **On** (ImageNet ön-eğitimli ağırlıklar için).

Çalıştırınca önce DICOM'lar `/kaggle/working/cache` içine önbelleğe alınır (~20–30 dk), sonra eğitim başlar (~15 epoch).
Bitince **Output** sekmesinden `rsna_lumbar.pth` ve `model_config.json` dosyalarını indirip `mr_asistan/models/` klasörüne koyun.

Hızlı deneme için son hücrede `sys.argv` satırına `--max-studies 100 --epochs 2` ekleyebilirsiniz.

In [ ]:
"""Train LumbarNet on the RSNA 2024 Lumbar Spine Degenerative Classification data.

Designed to run in a Kaggle notebook (GPU T4/P100, Internet ON for pretrained
weights). `build_notebook.py` inlines the shared core code so the notebook is
self-contained. Outputs (in --out-dir):
  rsna_lumbar.pth     model weights
  model_config.json   backbone, calibration temperature and validation metrics
"""

from __future__ import annotations

import argparse
import json
import math
import random
import time
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# ===== core/constants.py =====
"""Shared constants for training (Kaggle) and local inference.

The label layout follows the RSNA 2024 Lumbar Spine Degenerative
Classification competition: 5 conditions x 5 levels = 25 targets, each
graded Normal/Mild, Moderate or Severe.
"""

LEVELS = ["l1_l2", "l2_l3", "l3_l4", "l4_l5", "l5_s1"]
LEVEL_LABELS = {"l1_l2": "L1-L2", "l2_l3": "L2-L3", "l3_l4": "L3-L4", "l4_l5": "L4-L5", "l5_s1": "L5-S1"}

CONDITIONS = [
    "spinal_canal_stenosis",
    "left_neural_foraminal_narrowing",
    "right_neural_foraminal_narrowing",
    "left_subarticular_stenosis",
    "right_subarticular_stenosis",
]
CONDITION_LABELS_TR = {
    "spinal_canal_stenosis": "Spinal kanal darlığı",
    "left_neural_foraminal_narrowing": "Sol nöral foraminal darlık",
    "right_neural_foraminal_narrowing": "Sağ nöral foraminal darlık",
    "left_subarticular_stenosis": "Sol subartiküler (lateral reses) darlık",
    "right_subarticular_stenosis": "Sağ subartiküler (lateral reses) darlık",
}

# Target order: condition-major, same as the RSNA train.csv columns.
TARGETS = [f"{c}_{l}" for c in CONDITIONS for l in LEVELS]

SEVERITIES = ["Normal/Mild", "Moderate", "Severe"]
SEVERITY_LABELS_TR = ["Normal/Hafif", "Orta", "Ağır"]
SEVERITY_WEIGHTS = [1.0, 2.0, 4.0]  # competition metric weights

# Input series ("roles") and which condition is best seen on which series.
SERIES_ROLES = ["sag_t1", "sag_t2", "ax_t2"]
SERIES_ROLE_LABELS = {"sag_t1": "Sagittal T1", "sag_t2": "Sagittal T2/STIR", "ax_t2": "Axial T2"}
RSNA_DESCRIPTION_TO_ROLE = {"Sagittal T1": "sag_t1", "Sagittal T2/STIR": "sag_t2", "Axial T2": "ax_t2"}
CONDITION_TO_ROLE = {
    "spinal_canal_stenosis": "sag_t2",
    "left_neural_foraminal_narrowing": "sag_t1",
    "right_neural_foraminal_narrowing": "sag_t1",
    "left_subarticular_stenosis": "ax_t2",
    "right_subarticular_stenosis": "ax_t2",
}

SLICES_PER_SERIES = {"sag_t1": 10, "sag_t2": 10, "ax_t2": 10}
IMAGE_SIZE = 256


# ===== core/dicom_io.py =====
"""DICOM loading and preprocessing shared by training and inference.

Every series is turned into a fixed-size uint8 stack of shape
(n_slices, IMAGE_SIZE, IMAGE_SIZE). Training and inference must use exactly
this code, otherwise the model sees different inputs at serving time.
"""


from collections import defaultdict
from pathlib import Path

import numpy as np
import pydicom
from PIL import Image



def _slice_position(ds) -> float:
    """Position of a slice along its stack normal (falls back to InstanceNumber)."""
    try:
        orient = np.array(ds.ImageOrientationPatient, dtype=float)
        pos = np.array(ds.ImagePositionPatient, dtype=float)
        normal = np.cross(orient[:3], orient[3:])
        return float(np.dot(pos, normal))
    except (AttributeError, ValueError, TypeError):
        return float(getattr(ds, "InstanceNumber", 0) or 0)


def sort_series_files(paths: list[Path]) -> list[Path]:
    headers = []
    for p in paths:
        try:
            ds = pydicom.dcmread(str(p), stop_before_pixels=True)
        except Exception:
            continue
        headers.append((_slice_position(ds), p))
    headers.sort(key=lambda t: t[0])
    return [p for _, p in headers]


def _normalize_to_uint8(stack: np.ndarray) -> np.ndarray:
    lo, hi = np.percentile(stack, [1, 99])
    if hi <= lo:
        hi = lo + 1.0
    stack = np.clip((stack - lo) / (hi - lo), 0.0, 1.0)
    return (stack * 255).astype(np.uint8)


def load_series_stack(paths: list[Path], n_slices: int) -> np.ndarray:
    """Read a series, pick n evenly spaced slices, resize and normalize."""
    if not paths:
        return np.zeros((n_slices, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
    ordered = sort_series_files(paths)
    if not ordered:
        return np.zeros((n_slices, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
    idx = np.linspace(0, len(ordered) - 1, n_slices).round().astype(int)
    slices = []
    for i in idx:
        arr = pydicom.dcmread(str(ordered[i])).pixel_array.astype(np.float32)
        if arr.ndim == 3:  # multi-frame / RGB edge cases: take first plane
            arr = arr[..., 0] if arr.shape[-1] in (3, 4) else arr[0]
        img = Image.fromarray(arr).resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
        slices.append(np.asarray(img, dtype=np.float32))
    return _normalize_to_uint8(np.stack(slices))


def load_study_stacks(series_paths: dict[str, list[Path]]) -> dict[str, np.ndarray]:
    """series_paths maps role -> list of dicom files (missing roles become zeros)."""
    return {role: load_series_stack(series_paths.get(role, []), SLICES_PER_SERIES[role]) for role in SERIES_ROLES}


# --- Series discovery for uploaded studies -------------------------------------------------------


def group_by_series(root: Path) -> dict[str, dict]:
    """Group every DICOM under root by SeriesInstanceUID and collect header hints."""
    groups: dict[str, dict] = defaultdict(lambda: {"files": [], "header": None})
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        try:
            ds = pydicom.dcmread(str(p), stop_before_pixels=True)
        except Exception:
            continue
        uid = str(getattr(ds, "SeriesInstanceUID", "") or p.parent.name)
        g = groups[uid]
        g["files"].append(p)
        if g["header"] is None:
            g["header"] = ds
    return dict(groups)


def guess_series_role(ds) -> str | None:
    """Best-effort guess of sag_t1 / sag_t2 / ax_t2 from DICOM headers."""
    desc = str(getattr(ds, "SeriesDescription", "") or "").lower()
    plane = None
    try:
        orient = np.array(ds.ImageOrientationPatient, dtype=float)
        axis = int(np.argmax(np.abs(np.cross(orient[:3], orient[3:]))))
        plane = {0: "sag", 1: "cor", 2: "ax"}[axis]
    except (AttributeError, ValueError, TypeError):
        if "sag" in desc:
            plane = "sag"
        elif "ax" in desc or "tra" in desc:
            plane = "ax"

    weighting = None
    if "stir" in desc or "t2" in desc:
        weighting = "t2"
    elif "t1" in desc:
        weighting = "t1"
    else:
        te = float(getattr(ds, "EchoTime", 0) or 0)
        tr = float(getattr(ds, "RepetitionTime", 0) or 0)
        if te >= 60:
            weighting = "t2"
        elif 0 < te < 30 and 0 < tr < 1000:
            weighting = "t1"

    if plane == "sag" and weighting in ("t1", "t2"):
        return f"sag_{weighting}"
    if plane == "ax" and weighting != "t1":
        return "ax_t2"
    return None


def describe_series(groups: dict[str, dict]) -> list[dict]:
    out = []
    for uid, g in groups.items():
        ds = g["header"]
        out.append(
            {
                "series_uid": uid,
                "description": str(getattr(ds, "SeriesDescription", "") or ""),
                "n_files": len(g["files"]),
                "guessed_role": guess_series_role(ds),
            }
        )
    out.sort(key=lambda s: (s["guessed_role"] is None, s["guessed_role"] or "", -s["n_files"]))
    return out


# ===== core/model.py =====
"""Three-branch 2.5D CNN: one backbone per MRI series, slices stacked as channels.

Each branch sees one series (sag T1, sag T2/STIR, ax T2). Branch features are
concatenated and mapped to 25 targets x 3 severities. Keeping branches
separate lets Grad-CAM show *where on which series* the model looked.
"""


import numpy as np
import timm
import torch
import torch.nn as nn


DEFAULT_BACKBONE = "efficientnet_b0"


class LumbarNet(nn.Module):
    def __init__(self, backbone: str = DEFAULT_BACKBONE, pretrained: bool = False, dropout: float = 0.2):
        super().__init__()
        self.branches = nn.ModuleDict(
            {
                role: timm.create_model(
                    backbone, pretrained=pretrained, in_chans=SLICES_PER_SERIES[role], num_classes=0, global_pool=""
                )
                for role in SERIES_ROLES
            }
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        feat = self.branches[SERIES_ROLES[0]].num_features
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat * len(SERIES_ROLES), 512),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(512, len(TARGETS) * len(SEVERITIES)),
        )
        # Filled by forward() so Grad-CAM can read the last conv maps per branch.
        self.feature_maps: dict[str, torch.Tensor] = {}

    def forward(self, inputs: dict[str, torch.Tensor]) -> torch.Tensor:
        pooled = []
        for role in SERIES_ROLES:
            fmap = self.branches[role](inputs[role])
            if fmap.requires_grad:
                fmap.retain_grad()
            self.feature_maps[role] = fmap
            pooled.append(self.pool(fmap).flatten(1))
        logits = self.head(torch.cat(pooled, dim=1))
        return logits.view(-1, len(TARGETS), len(SEVERITIES))


def stacks_to_tensors(stacks: dict[str, np.ndarray], device: str = "cpu") -> dict[str, torch.Tensor]:
    """uint8 (C,H,W) stacks -> float tensors (1,C,H,W) in [0,1]."""
    return {role: torch.from_numpy(stacks[role]).float().div(255.0).unsqueeze(0).to(device) for role in SERIES_ROLES}


KAGGLE_DATA = "/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification"


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data-dir", default=KAGGLE_DATA)
    p.add_argument("--out-dir", default="/kaggle/working")
    p.add_argument("--cache-dir", default="/kaggle/working/cache")
    p.add_argument("--backbone", default=DEFAULT_BACKBONE)
    p.add_argument("--epochs", type=int, default=15)
    p.add_argument("--batch-size", type=int, default=8)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--val-frac", type=float, default=0.2)
    p.add_argument("--max-studies", type=int, default=0, help="debug: limit number of studies")
    p.add_argument("--num-workers", type=int, default=4)
    p.add_argument("--no-pretrained", action="store_true")
    p.add_argument("--seed", type=int, default=42)
    args, _ = p.parse_known_args()  # ignore Jupyter's own argv
    return args


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# --- Data preparation --------------------------------------------------------------------------


def build_labels(train_csv: pd.DataFrame) -> np.ndarray:
    sev_to_idx = {s: i for i, s in enumerate(SEVERITIES)}
    labels = np.full((len(train_csv), len(TARGETS)), -100, dtype=np.int64)
    for j, t in enumerate(TARGETS):
        col = train_csv[t].map(sev_to_idx)
        labels[:, j] = col.fillna(-100).astype(np.int64).values
    return labels


def study_series_paths(data_dir: Path, study_id: int, desc: pd.DataFrame) -> dict[str, list[Path]]:
    out: dict[str, list[Path]] = {}
    for _, row in desc[desc.study_id == study_id].iterrows():
        role = RSNA_DESCRIPTION_TO_ROLE.get(row.series_description)
        if role is None or role in out:  # keep the first series of each type
            continue
        series_dir = data_dir / "train_images" / str(study_id) / str(row.series_id)
        out[role] = sorted(series_dir.glob("*.dcm"))
    return out


def _cache_one(job):
    data_dir, cache_dir, study_id, desc = job
    target = cache_dir / f"{study_id}.npz"
    if target.exists():
        return study_id, True
    try:
        stacks = load_study_stacks(study_series_paths(data_dir, study_id, desc))
        np.savez(target, **stacks)
        return study_id, True
    except Exception as e:  # corrupt/odd studies are skipped, not fatal
        print(f"[cache] study {study_id} failed: {e}")
        return study_id, False


def build_cache(data_dir: Path, cache_dir: Path, study_ids: list[int], desc: pd.DataFrame, workers: int) -> set[int]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    jobs = [(data_dir, cache_dir, sid, desc) for sid in study_ids]
    ok = set()
    t0 = time.time()
    with Pool(max(1, workers)) as pool:
        for i, (sid, success) in enumerate(pool.imap_unordered(_cache_one, jobs, chunksize=4), 1):
            if success:
                ok.add(sid)
            if i % 100 == 0 or i == len(jobs):
                print(f"[cache] {i}/{len(jobs)} studies ({time.time() - t0:.0f}s)")
    return ok


# --- Dataset & augmentation --------------------------------------------------------------------


def random_affine(x: torch.Tensor, max_rot=10, max_scale=0.1, max_shift=0.08) -> torch.Tensor:
    """Same random affine for all slices of one series. x: (C,H,W). No flips (left/right labels)."""
    ang = math.radians(random.uniform(-max_rot, max_rot))
    s = 1.0 + random.uniform(-max_scale, max_scale)
    tx, ty = random.uniform(-max_shift, max_shift), random.uniform(-max_shift, max_shift)
    theta = torch.tensor(
        [[math.cos(ang) / s, -math.sin(ang) / s, tx], [math.sin(ang) / s, math.cos(ang) / s, ty]], dtype=x.dtype
    ).unsqueeze(0)
    grid = F.affine_grid(theta, (1, *x.shape), align_corners=False)
    return F.grid_sample(x.unsqueeze(0), grid, align_corners=False, padding_mode="zeros").squeeze(0)


def random_intensity(x: torch.Tensor) -> torch.Tensor:
    gamma = random.uniform(0.8, 1.25)
    x = x.clamp(0, 1) ** gamma
    return (x * random.uniform(0.9, 1.1) + random.uniform(-0.05, 0.05)).clamp(0, 1)


class StudyDataset(Dataset):
    def __init__(self, study_ids, labels, cache_dir: Path, train: bool):
        self.study_ids, self.labels, self.cache_dir, self.train = list(study_ids), labels, cache_dir, train

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, i):
        data = np.load(self.cache_dir / f"{self.study_ids[i]}.npz")
        x = {}
        for role in SERIES_ROLES:
            t = torch.from_numpy(data[role]).float() / 255.0
            if self.train:
                if random.random() < 0.8:
                    t = random_affine(t)
                if random.random() < 0.8:
                    t = random_intensity(t)
            x[role] = t
        return x, torch.from_numpy(self.labels[i])


# --- Metrics -----------------------------------------------------------------------------------


def weighted_log_loss(logits: torch.Tensor, labels: torch.Tensor, temperature: float = 1.0) -> float:
    w = torch.tensor(SEVERITY_WEIGHTS, device=logits.device)
    return F.cross_entropy(
        (logits / temperature).reshape(-1, 3), labels.reshape(-1), weight=w, ignore_index=-100
    ).item()


def auroc(y_true: np.ndarray, y_score: np.ndarray) -> float | None:
    """Rank-based AUROC (Mann-Whitney U), no sklearn needed."""
    pos, neg = y_true == 1, y_true == 0
    n_pos, n_neg = pos.sum(), neg.sum()
    if n_pos == 0 or n_neg == 0:
        return None
    order = np.argsort(y_score)
    ranks = np.empty(len(y_score), dtype=float)
    ranks[order] = np.arange(1, len(y_score) + 1)
    # average ranks for ties
    for v in np.unique(y_score):
        m = y_score == v
        if m.sum() > 1:
            ranks[m] = ranks[m].mean()
    return float((ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def fit_temperature(logits: torch.Tensor, labels: torch.Tensor) -> float:
    """Temperature scaling on the validation set so probabilities are better calibrated."""
    log_t = torch.zeros(1, requires_grad=True, device=logits.device)
    w = torch.tensor(SEVERITY_WEIGHTS, device=logits.device)
    opt = torch.optim.LBFGS([log_t], lr=0.05, max_iter=200)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(
            (logits / log_t.exp()).reshape(-1, 3), labels.reshape(-1), weight=w, ignore_index=-100
        )
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp().item())


def condition_metrics(logits: torch.Tensor, labels: torch.Tensor, temperature: float) -> dict:
    probs = torch.softmax(logits / temperature, dim=-1).cpu().numpy()
    y = labels.cpu().numpy()
    out = {}
    for ci, cond in enumerate(CONDITIONS):
        sl = slice(ci * len(LEVELS), (ci + 1) * len(LEVELS))
        p, t = probs[:, sl].reshape(-1, 3), y[:, sl].reshape(-1)
        valid = t >= 0
        p, t = p[valid], t[valid]
        out[cond] = {
            "auroc_moderate_or_severe": auroc((t >= 1).astype(int), p[:, 1] + p[:, 2]),
            "auroc_severe": auroc((t == 2).astype(int), p[:, 2]),
            "n": int(valid.sum()),
            "prevalence_moderate_or_severe": float((t >= 1).mean()) if len(t) else None,
        }
    return out


# --- Training loop -----------------------------------------------------------------------------


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    for x, y in loader:
        x = {k: v.to(device, non_blocking=True) for k, v in x.items()}
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(x)
        all_logits.append(logits.float())
        all_labels.append(y.to(device))
    return torch.cat(all_logits), torch.cat(all_labels)


def main():
    args = parse_args()
    seed_everything(args.seed)
    data_dir, out_dir, cache_dir = Path(args.data_dir), Path(args.out_dir), Path(args.cache_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device={device}  backbone={args.backbone}")

    train_csv = pd.read_csv(data_dir / "train.csv")
    desc = pd.read_csv(data_dir / "train_series_descriptions.csv")
    if args.max_studies:
        train_csv = train_csv.head(args.max_studies)

    ok = build_cache(data_dir, cache_dir, train_csv.study_id.tolist(), desc, args.num_workers)
    train_csv = train_csv[train_csv.study_id.isin(ok)].reset_index(drop=True)
    labels = build_labels(train_csv)
    print(f"studies with cached images: {len(train_csv)}")

    idx = np.random.permutation(len(train_csv))
    n_val = max(1, int(len(idx) * args.val_frac))
    val_idx, tr_idx = idx[:n_val], idx[n_val:]
    ids = train_csv.study_id.values
    dl_kw = dict(batch_size=args.batch_size, num_workers=args.num_workers, pin_memory=device.type == "cuda")
    train_dl = DataLoader(StudyDataset(ids[tr_idx], labels[tr_idx], cache_dir, True), shuffle=True, drop_last=True, **dl_kw)
    val_dl = DataLoader(StudyDataset(ids[val_idx], labels[val_idx], cache_dir, False), shuffle=False, **dl_kw)

    pretrained = not args.no_pretrained
    try:
        model = LumbarNet(args.backbone, pretrained=pretrained)
    except Exception as e:
        print(f"!! pretrained weights could not be loaded ({e}); is Internet enabled? training from scratch")
        pretrained = False
        model = LumbarNet(args.backbone, pretrained=False)
    model.to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-2)
    total_steps = max(1, args.epochs * len(train_dl))
    warmup = max(1, int(0.05 * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total_steps)))
    )
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")
    loss_w = torch.tensor(SEVERITY_WEIGHTS, device=device)

    best_loss, best_state = float("inf"), None
    for epoch in range(1, args.epochs + 1):
        model.train()
        t0, running = time.time(), 0.0
        for step, (x, y) in enumerate(train_dl, 1):
            x = {k: v.to(device, non_blocking=True) for k, v in x.items()}
            y = y.to(device)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits = model(x)
            loss = F.cross_entropy(logits.float().reshape(-1, 3), y.reshape(-1), weight=loss_w, ignore_index=-100)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += loss.item()
        val_logits, val_labels = predict(model, val_dl, device)
        val_loss = weighted_log_loss(val_logits, val_labels)
        print(
            f"epoch {epoch}/{args.epochs}  train_loss={running / max(1, len(train_dl)):.4f}  "
            f"val_wlogloss={val_loss:.4f}  ({time.time() - t0:.0f}s)"
        )
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    val_logits, val_labels = predict(model, val_dl, device)
    temperature = fit_temperature(val_logits, val_labels)
    metrics = {
        "val_weighted_logloss": weighted_log_loss(val_logits, val_labels),
        "val_weighted_logloss_calibrated": weighted_log_loss(val_logits, val_labels, temperature),
        "per_condition": condition_metrics(val_logits, val_labels, temperature),
        "n_train": int(len(tr_idx)),
        "n_val": int(len(val_idx)),
    }
    print(json.dumps(metrics, indent=2))

    torch.save(best_state, out_dir / "rsna_lumbar.pth")
    config = {
        "backbone": args.backbone,
        "pretrained_init": pretrained,
        "temperature": temperature,
        "image_size": IMAGE_SIZE,
        "slices_per_series": SLICES_PER_SERIES,
        "targets": TARGETS,
        "severities": SEVERITIES,
        "epochs": args.epochs,
        "trained_at": time.strftime("%Y-%m-%d %H:%M"),
        "dataset": "RSNA 2024 Lumbar Spine Degenerative Classification",
        "metrics": metrics,
    }
    (out_dir / "model_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    print(f"saved -> {out_dir / 'rsna_lumbar.pth'} and model_config.json")




In [ ]:
import sys
sys.argv = ['train']  # ör: ['train', '--epochs', '15']
main()
